In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('fake_job_postings.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nClass distribution:")
print(df['fraudulent'].value_counts())
print(f"\nFake %: {df['fraudulent'].mean()*100:.2f}%")
print("\nNull counts:")
print(df.isnull().sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
df['fraudulent'].value_counts().plot(
    kind='bar', ax=axes[0],
    color=["#1a099a", "#ad1807"],
    edgecolor='black'
)
axes[0].set_title('Real vs Fake Job Postings')
axes[0].set_xticklabels(['Real (0)', 'Fake (1)'], rotation=0)
axes[0].set_ylabel('Count')

# Pie chart
df['fraudulent'].value_counts().plot(
    kind='pie', ax=axes[1],
    labels=['Real', 'Fake'],
    colors=["#1a099a", "#ad1807"],
    autopct='%1.1f%%',
    startangle=90
)
axes[1].set_title('Class Balance')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()
print("Saved: class_distribution.png")

In [ ]:
structured_cols = ['has_company_logo', 'telecommuting', 'salary_range','company_profile', 'requirements', 'benefits']

print("=== Fake Rate by Structured Feature ===\n")

# Salary range
df['has_salary'] = df['salary_range'].notna().astype(int)
print("Has Salary:")
print(df.groupby('has_salary')['fraudulent'].mean().rename({0:'No salary', 1:'Has salary'}))

# Company logo
print("\nHas Company Logo:")
print(df.groupby('has_company_logo')['fraudulent'].mean())

# Company profile
df['has_profile'] = df['company_profile'].notna().astype(int)
print("\nHas Company Profile:")
print(df.groupby('has_profile')['fraudulent'].mean())

In [ ]:
print("=" * 50)
print("SUMMARY — Fake Rate by Feature Presence")
print("=" * 50)

summary = pd.DataFrame({
    'Feature': [
        'Has Salary Range',
        'Has Company Logo',
        'Has Company Profile',
    ],
    'Fake Rate (Absent)': [
        df[df['has_salary']==0]['fraudulent'].mean(),
        df[df['has_company_logo']==0]['fraudulent'].mean(),
        df[df['has_profile']==0]['fraudulent'].mean(),
    ],
    'Fake Rate (Present)': [
        df[df['has_salary']==1]['fraudulent'].mean(),
        df[df['has_company_logo']==1]['fraudulent'].mean(),
        df[df['has_profile']==1]['fraudulent'].mean(),
    ]
}).round(4)

summary['Signal Strength'] = (
    summary['Fake Rate (Absent)'] - summary['Fake Rate (Present)']
).round(4)

summary = summary.sort_values('Signal Strength', ascending=False)
print(summary.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

features   = summary['Feature'].values
absent     = summary['Fake Rate (Absent)'].values
present    = summary['Fake Rate (Present)'].values

x      = np.arange(len(features))
width  = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, absent,  width, label='Feature Absent', color='#e74c3c', alpha=0.85)
bars2 = ax.bar(x + width/2, present, width, label='Feature Present', color='#2ecc71', alpha=0.85)

ax.set_xlabel('Feature')
ax.set_ylabel('Fake Job Rate')
ax.set_title('Fake Rate: Feature Absent vs Present')
ax.set_xticks(x)
ax.set_xticklabels(features, rotation=25, ha='right')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('structured_feature_analysis.png', dpi=150)
plt.show()
print("Saved: structured_feature_analysis.png")

In [ ]:
df['desc_length'] = df['description'].fillna('').apply(len)

fig, ax = plt.subplots(figsize=(10, 4))
df[df['fraudulent']==0]['desc_length'].hist(
    bins=50, alpha=0.6, label='Real', color='#1a099a', ax=ax)
df[df['fraudulent']==1]['desc_length'].hist(
    bins=50, alpha=0.6, label='Fake', color='#ad1807', ax=ax)
ax.set_xlabel('Description Length (characters)')
ax.set_ylabel('Count')
ax.set_title('Description Length: Real vs Fake')
ax.legend()
plt.tight_layout()
plt.savefig('desc_length_distribution.png', dpi=150)
plt.show()

print("Real avg desc length:", df[df['fraudulent']==0]['desc_length'].mean().round(0))
print("Fake avg desc length:", df[df['fraudulent']==1]['desc_length'].mean().round(0))


# ── CELL 6: Top Industries / Job Functions ───────────────────
print("=== Top Industries (Fake Jobs) ===")
print(df[df['fraudulent']==1]['industry'].value_counts().head(10))

print("\n=== Top Job Functions (Fake Jobs) ===")
print(df[df['fraudulent']==1]['function'].value_counts().head(10))

In [ ]:
from wordcloud import WordCloud

real_text = ' '.join(df[df['fraudulent']==0]['description'].fillna(''))
fake_text = ' '.join(df[df['fraudulent']==1]['description'].fillna(''))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

WordCloud(width=800, height=400, background_color='white',colormap='Greens').generate(real_text).to_image()
axes[0].imshow(WordCloud(width=800, height=400,background_color='white', colormap='Greens').generate(real_text))
axes[0].set_title('Real Job Postings — Common Words')
axes[0].axis('off')

axes[1].imshow(WordCloud(width=800, height=400,background_color='white', colormap='Reds').generate(fake_text))
axes[1].set_title('Fake Job Postings — Common Words')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('wordcloud_comparison.png', dpi=150)
plt.show()